<img src=https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_universidad_icesi.svg width=300>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastianb92/ecomarket-rag-solution/blob/main/notebooks/EcoMarket_AI_RAG_Solution.ipynb)


# Maestría en Inteligencia Artificial  
## IA Generativa
---
# EcoMarket AI Support — Taller Práctico #2

**Integrantes:**  
- Johan Sebastian Bonilla  
- Edwin Gómez  

## Optimización de Atención al Cliente con IA Generativa (RAG) para asistente de e-commerce

En este notebook se desarrolla un chatbot inteligente orientado a responder consultas de clientes en un entorno de comercio electrónico, específicamente sobre estado de pedidos y procesos de devolución.

A diferencia de los enfoques tradicionales de generación de texto, este sistema utiliza la técnica de Retrieval-Augmented Generation (RAG), que combina la recuperación de información con modelos generativos de lenguaje.

El modelo no se limita a generar respuestas basadas únicamente en su conocimiento previo. En su lugar, primero recupera información relevante desde una base de conocimiento que incluye datos de pedidos y políticas de la tienda. Posteriormente, utiliza este contexto para generar respuestas más precisas, coherentes y alineadas con la información real del negocio.



Configura rutas y detecta si el entorno es Colab o local.

In [ ]:
import warnings
from importlib import metadata

warnings.filterwarnings('ignore')

installed_packages = {dist.metadata['Name'].lower() for dist in metadata.distributions() if dist.metadata.get('Name')}
IN_COLAB = 'google-colab' in installed_packages

Descarga dependencias y las instala automáticamente en Colab.

In [ ]:
if IN_COLAB:
    !wget -q https://raw.githubusercontent.com/sebastianb92/ecomarket-rag-solution/main/requirements.txt -O requirements.txt && \
    uv pip install -r requirements.txt
else:
    print("Entorno local — instala con: uv pip install -r requirements.txt")

IMPORTACIONES

In [ ]:
# Librerías estándar
import os
from pathlib import Path

# Datos
import pandas as pd

# Entorno
from google.colab import userdata

# Modelos
from langchain_groq import ChatGroq

# Embeddings y vector store
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata

# Carga de documentos
from langchain_community.document_loaders import (
    DataFrameLoader,
    PyPDFLoader,
    JSONLoader
)

# Procesamiento de texto
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Prompts y cadenas
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import RetrievalQA

Descarga los datos necesarios desde GitHub en Colab si no existen; en local usa los archivos ya disponibles.

In [ ]:
if IN_COLAB:
    !mkdir -p data

    # JSON
    ![ ! -f data/FAQ.json ] && wget -q -O data/FAQ.json \
    https://raw.githubusercontent.com/sebastianb92/ecomarket-rag-solution/main/data/FAQ.json

    # PDF
    ![ ! -f data/politica_devoluciones.pdf ] && wget -q -O data/politica_devoluciones.pdf \
    "https://raw.githubusercontent.com/sebastianb92/ecomarket-rag-solution/main/data/POL%C3%8DTICA%20DE%20DEVOLUCIONES.pdf"

    # Excel
    ![ ! -f data/pedidos_ecomarket.xlsx ] && wget -q -O data/pedidos_ecomarket.xlsx \
    https://raw.githubusercontent.com/sebastianb92/ecomarket-rag-solution/main/data/pedidos_ecomarket.xlsx

    print("Datos descargados desde GitHub")

else:
    print(f"Datos locales en: {DATA_DIR}")

## Implementación de RAG

En esta sección se construye el pipeline completo de Retrieval-Augmented Generation (RAG), integrando los diferentes componentes necesarios para recuperar información relevante de EcoMarket y generar respuestas en lenguaje natural.

### Componentes del sistema

1. **LLM (GROQ):**  `llama-3.3-70b-versatile`

   Modelo generativo encargado de producir la respuesta final en lenguaje natural, utilizando como contexto los documentos recuperados.

2. **Embeddings:** `intfloat/multilingual-e5-large`  
   Modelo avanzado de Hugging Face diseñado para tareas de recuperación semántica. Es multilingüe, lo que permite trabajar con documentos en inglés y consultas en español.  Genera vectores de 1024 dimensiones, lo que incrementa su capacidad semántica a costa de un mayor uso de memoria.
   
3. **Document Store (Vector Store):**  
   Se utiliza CHROMA (a través de LangChain), una librería optimizada para la indexación y búsqueda eficiente de vectores en espacios de alta dimensión.


Configura el modelo **LLM** de Groq con la API key y parámetros de generación.

In [ ]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=userdata.get("GROQ_API_KEY"),
    temperature=0.3 #Respuestas más deterministas (menos creativas)
)

Inicializa el modelo de **embeddings** para representar texto como vectores.

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={"device": "cuda"},  # cpu o "cuda"
    encode_kwargs={"normalize_embeddings": True}
)


Crea y configura la **vector store** para almacenar y consultar embeddings.

In [ ]:
vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",
)

## Indexación de datos

En esta sección se prepara la información para que pueda ser buscada eficientemente (base de un sistema RAG).

El proceso de indexación sigue tres pasos principales:

**Load (Carga):** Se cargan los datos desde diferentes fuentes como JSON, PDF o Excel usando loaders.

**Split (Fragmentación):** Los documentos se dividen en fragmentos más pequeños para facilitar su procesamiento y mejorar la búsqueda.

**Store (Almacenamiento):** Los fragmentos se convierten en embeddings y se almacenan en una base vectorial para permitir búsquedas semánticas posteriores.

Fuente: https://docs.langchain.com/oss/python/langchain/rag#huggingface

<img src="https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_indexing.png?w=840&fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=1838328a870c7353c42bf1cc2290a779" width="900">

Carga un archivo Excel, convierte cada fila en texto y la transforma en documentos para el sistema RAG.

In [ ]:
df = pd.read_excel("/content/data/pedidos_ecomarket.xlsx")

df["contenido"] = df.astype(str).agg(" | ".join, axis=1)

loader = DataFrameLoader(df, page_content_column="contenido")
docs_excel = loader.load()

Carga el archivo PDF y lo convierte en documentos procesables para el RAG.

In [ ]:
loader = PyPDFLoader("/content/data/politica_devoluciones.pdf")
docs_pdf = loader.load()

Carga el archivo JSON y transforma cada registro en texto estructurado para el RAG.

In [ ]:
loader = JSONLoader(
    file_path="/content/data/FAQ.json",
    jq_schema='.faq[] | "Categoría: \(.categoria)\nPregunta: \(.pregunta)\nRespuesta: \(.respuesta)"',
    text_content=True
)

docs_json = loader.load()

Une todos los documentos en una sola colección para su procesamiento.

In [ ]:
docs = docs_excel + docs_pdf + docs_json

Divide los documentos en fragmentos más pequeños para mejorar la búsqueda y el procesamiento.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Se dividieron los documentos en {len(all_splits)} sub-documentos.")

Limpia metadatos complejos de los documentos para evitar errores al almacenarlos.

In [ ]:
all_splits = filter_complex_metadata(all_splits)

Agrega los documentos a la base vectorial y muestra algunos IDs generados.

In [ ]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

## Recuperación y generación

En esta etapa se construye la lógica de respuesta del sistema RAG.

**Retrieve (Recuperación):** A partir de la pregunta del usuario, se buscan los fragmentos más relevantes en la base vectorial usando un retriever.

**Generate (Generación):** Un modelo genera la respuesta utilizando un prompt que combina la pregunta con la información recuperada.

Fuente: https://docs.langchain.com/oss/python/langchain/rag#huggingface

<img src="https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_retrieval_generation.png?w=840&fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=67fe2302e241fc24238a5df1cf56573d" width="900">

Configura el **retriever** para buscar los 3 documentos más similares en la base vectorial.

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

Este prompt define a EcoBot, un asistente de e-commerce que responde consultas sobre estado de pedidos y devoluciones usando información de un contexto (RAG).

Primero clasifica la intención del usuario y luego aplica un flujo específico. Incluye una regla crítica: solo usa información de pedidos si el usuario proporciona explícitamente un número de pedido, evitando errores comunes de asociación automática entre productos y pedidos.


In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres EcoBot, asistente de e-commerce.

PERSONALIDAD: Amable, honesto, empático y proactivo.

OBJETIVO:
Resolver consultas sobre:
1) Estado de pedidos
2) Devoluciones

REGLAS GENERALES:
- Usa SOLO el contexto proporcionado
- NO inventes información
- Si no encuentras datos, dilo claramente
- Responde SIEMPRE en español
- Tono amable, claro y profesional
- Sé conciso pero útil

--------------------------------------------------
PASO 0 — IDENTIFICAR INTENCIÓN
--------------------------------------------------
Clasifica la pregunta del usuario en UNA de estas categorías:

A) ESTADO_PEDIDO → si pregunta por envío, entrega, tracking, estado
B) DEVOLUCION → si pregunta por devolver, reembolso, cambios
C) AMBIGUO → si no está claro

- Si es AMBIGUO → pide aclaración breve antes de continuar

--------------------------------------------------
REGLA DE PRIORIDAD (CRÍTICA - OBLIGATORIA)
--------------------------------------------------

Cuando la consulta sea sobre DEVOLUCIONES:

- SOLO puedes usar un pedido si el usuario menciona explícitamente un número de pedido
  (ej: ECO-12345, pedido 123, etc.)

- PROHIBIDO:
  - Usar pedidos encontrados en el contexto si el usuario NO los mencionó
  - Inferir pedidos a partir del producto
  - Relacionar automáticamente producto ↔ pedido

- Si el usuario menciona SOLO un producto (ej: "set de cubiertos de bambú"):
  → IGNORA completamente cualquier pedido en el contexto
  → Responde como CONSULTA GENERAL (B2)

Esta regla tiene prioridad sobre cualquier otra instrucción.

--------------------------------------------------
FLUJO A — ESTADO DEL PEDIDO
--------------------------------------------------

PASO 1 — Buscar número de pedido en el contexto

PASO 2 — RESPUESTA

SI el pedido EXISTE:
- Explica el estado claramente

FORMATO OBLIGATORIO:

Hola [Nombre si está disponible, si no usa "Hola"],

Aquí tienes la información de tu pedido:

- Estado:
- Producto:
- Fecha estimada:
- Tracking:
- Última ubicación:
- Observaciones:

REGLAS ADICIONALES:
- Si está RETRASADO:
  → Empieza con disculpa sincera
  → Explica causa + nueva fecha

- Si está CANCELADO:
  → Explica estado del reembolso + plazo claro

- Si está PENDIENTE DE PAGO:
  → Indica acción requerida con tono amable

- Siempre indica el siguiente paso esperado

---

SI el pedido NO EXISTE:
- Indica que no se encontró el pedido
- Sugiere verificar el número
- Ofrece contacto:
  soporte@ecomarket.com
  900-ECO-MKT
- NUNCA inventes información

--------------------------------------------------
FLUJO B — DEVOLUCIONES
--------------------------------------------------

PASO 1 — IDENTIFICAR TIPO DE SOLICITUD

Clasifica en:

B1) DEVOLUCIÓN CON PEDIDO:
- SOLO si el usuario proporciona explícitamente un número de pedido

B2) CONSULTA GENERAL DE DEVOLUCIÓN:
- El usuario NO proporciona número de pedido
- Solo menciona un producto
- Pregunta por proceso, política o condiciones

--------------------------------------------------
CASO B1 — DEVOLUCIÓN CON PEDIDO
--------------------------------------------------

PASO 2 — Evaluar si aplica política usando el contexto

SI APLICA:

Hola [Nombre],

Claro, puedo ayudarte con la devolución de tu producto.

Sigue estos pasos:

1. [Paso 1]
2. [Paso 2]
3. [Paso 3]

- Plazo de reembolso:
- Método:

Puedes optar por crédito en tienda con 5% adicional.

---

SI NO APLICA:

Hola [Nombre],

1. Entiendo cómo te sientes con esta situación.
2. [Razón clara y humana]
3. [Alternativa concreta]

- Nunca respondas con un "no" sin alternativa

--------------------------------------------------
CASO B2 — CONSULTA GENERAL (SIN PEDIDO)
--------------------------------------------------

NO usar información de pedidos aunque exista en el contexto

FORMATO:

Hola,

Con gusto te explico cómo funciona nuestro proceso de devoluciones:

[Explicación clara basada SOLO en políticas del contexto]

Si deseas iniciar una devolución, necesitarás tu número de pedido.

¿Tienes el número de pedido? Con eso puedo ayudarte mejor.

--------------------------------------------------
CIERRE (SIEMPRE)
--------------------------------------------------

¿Puedo ayudarte con algo más?
"""),

    ("human", """Contexto:
{context}

Pregunta:
{question}
""")
])

Crea una cadena RAG que combina el modelo y el retriever para **generar** respuestas.

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt}
)

---
## Ejercicio 1 — Consultas de Estado de Pedido
Demuestra el manejo de diferentes estados del producto y la protección anti-alucinación.

Pedido EN TRÁNSITO

In [ ]:
query = "¿Quiero saber el estado de mi pedido ECO-12345?"
respuesta = qa_chain.run(query)

print(respuesta)

Pedido RETRASADO

In [ ]:
query = "¿Quiero saber el estado de mi pedido ECO-12346?"
respuesta = qa_chain.run(query)

print(respuesta)

Pedido ENTREGADO

In [ ]:
query = "¿Quiero saber el estado de mi pedido ECO-12347?"
respuesta = qa_chain.run(query)

print(respuesta)

Pedido CANCELADO

In [ ]:
query = "¿Quiero saber el estado de mi pedido ECO-12349?"
respuesta = qa_chain.run(query)

print(respuesta)

Pedido PENDIENTE DE PAGO

In [ ]:
query = "¿Quiero saber el estado de mi pedido ECO-12371?"
respuesta = qa_chain.run(query)

print(respuesta)

Número INEXISTENTE

In [ ]:
query = "¿Quiero saber el estado de mi pedido ECO-99999?"
respuesta = qa_chain.run(query)

print(respuesta)

## Ejercicio 2 — Solicitudes de Devolución
El desafío: responder con empatía tanto el «sí» como el «no»,
y nunca dar una negativa sin ofrecer una alternativa.

Devolución APROBADA

In [ ]:
query = "Quiero hacer la devolucion de mi Botella de acero inoxidable 750ml ya que el diseño no me gusta, ¿que debo hacer?"
respuesta = qa_chain.run(query)

print(respuesta)

Devolución RECHAZADA

In [ ]:
query = "Quiero hacer la devolucion de mi Jabón orgánico artesanal ya que no me gusta el olor, ¿que debo hacer?"
respuesta = qa_chain.run(query)

print(respuesta)

Devolución RECHAZADA

In [ ]:
query = "Quiero hacer la devolucion del Mix de frutos secos orgánicos ya que cambie de opinion y no lo quiero, ¿que debo hacer?"
respuesta = qa_chain.run(query)

print(respuesta)

Devolución APROBADA

In [ ]:
query = "Quiero hacer la devolucion del Cepillo de dientes de bambú ya que el paquete llegó aplastado y roto, ¿que debo hacer?"
respuesta = qa_chain.run(query)

print(respuesta)

Devolución FUERA DE PLAZO

In [ ]:
query = "Quiero hacer la devolucion del Set de cubiertos de bambú ya que me regalaron otro, ¿que debo hacer?"
respuesta = qa_chain.run(query)

print(respuesta)